# DLW 半离散系统的非线性化

从交错格点上的双线性方程出发，分别得到消去辅助势的两场形式与保留势的比值形式。中间的恒等变换以公式展开，Lean 中的起点和终点对应两条推导路线。最后将连续单孤子代入半离散方程，比较空间残差、二阶主系数及格距收敛。

证明所需的定义与定理来自已有的本地 Lean 证明库。

In [ ]:
# 载入非线性化证明所需的本地 Lean 环境。
import sys
sys.path.insert(0, "C:/Users/msz/aca/Workspaces/report_colab_20261006")

from report_runtime import load_lean
lean_notebook_session = load_lean()


## 1　DLW 半离散系统的非线性化

### 1.1　消去辅助势的两场形式

载入 (u, ω) 与 (u, v) 非线性化的定义、证明及已编译的证明库。


In [ ]:
%%lean uw import
import ReportNonlinearUW


考虑交错格点上的双线性方程

$$B_{a-h/2}F_j\cdot G_j=0,\qquad B_{a+h/2}F_j\cdot G_{j+1}=0,\qquad B_s=D_x^2+D_t+2sD_x.\tag{1}$$

在 Lean 中，两条双线性假设合并为 `StartPoint`：


In [ ]:
%%lean uw start
import StepUWImport

namespace DLWContract.ReportUW

def StartPoint (a h : ℝ) (F G : Lattice) : Prop :=
  ∀ j, bil (a-h/2) (F j) (G j) = 0 ∧
    bil (a+h/2) (F j) (G (j+1)) = 0

#print StartPoint

end DLWContract.ReportUW


其中 $D$ 为 Hirota 双线性算子，$F_j$ 与 $G_j$ 分别位于 $y=(j+\tfrac12)h$ 和 $y=jh$，$h\ne0$。在 $F_j,G_j$ 非零的区域内，令 $\alpha_j=\log F_j$、$\beta_j=\log G_j$。利用恒等式

$$\frac{B_sf\cdot g}{fg}=(\log f+\log g)_{xx}+(\log f-\log g)_x^2
+(\log f-\log g)_t+2s(\log f-\log g)_x,\tag{2}$$

将式（1）除以相应的 $\tau$ 函数乘积，得到

$$\begin{aligned}
A_j={}&(\alpha_j+\beta_j)_{xx}+(\alpha_j-\beta_j)_x^2+(\alpha_j-\beta_j)_t+(2a-h)(\alpha_j-\beta_j)_x=0,\\
C_j={}&(\alpha_j+\beta_{j+1})_{xx}+(\alpha_j-\beta_{j+1})_x^2+(\alpha_j-\beta_{j+1})_t+(2a+h)(\alpha_j-\beta_{j+1})_x=0.
\end{aligned}\tag{3}$$

取两侧对数差导数的和与差

$$u_j=(2\alpha_j-\beta_j-\beta_{j+1})_x,\qquad
\omega_j=(\beta_{j+1}-\beta_j)_x.\tag{4}$$

于是两侧对数差的导数分别为 $(u_j+\omega_j)/2$ 和 $(u_j-\omega_j)/2$。将式（3）相加、相减并对 $x$ 求导，记

$$Z_j=(2\alpha_j+\beta_j+\beta_{j+1})_x.\tag{5}$$

可得

$$\begin{aligned}
u_{j,t}+\partial_x\!\left[\frac{u_j^2+\omega_j^2}{2}+2au_j-h\omega_j\right]+Z_{j,xx}&=0,\\
\omega_{j,t}+[(u_j+2a)\omega_j-hu_j]_x-\omega_{j,xx}&=0.
\end{aligned}\tag{6}$$

定义 $\delta_-f_j=(f_j-f_{j-1})/h$、$M_-f_j=(f_j+f_{j-1})/2$。由式（4）有 $\delta_-Z_j=\delta_-u_j+(4/h)M_-\omega_j$，从而消去 $Z_j$，得到闭合的非线性半离散系统

$$\boxed{\begin{aligned}
\delta_-u_{j,t}+\partial_x\delta_-\!\left[\frac{u_j^2+\omega_j^2}{2}+2au_j-h\omega_j\right]
+\partial_x^2\left(\delta_-u_j+\frac4hM_-\omega_j\right)&=0,\\
\omega_{j,t}+\partial_x[(u_j+2a)\omega_j-hu_j]-\omega_{j,xx}&=0.
\end{aligned}}\tag{7}$$

为恢复连续 DLW 的物理变量，令 $v_j=(4/h)\omega_j+\delta_0u_j$，其中 $\delta_0f_j=(f_{j+1}-f_{j-1})/(2h)$。写 $W_j=v_j-\delta_0u_j$、$\Delta_hf_j=(f_{j+1}-2f_j+f_{j-1})/h^2$，式（7）等价地表示为

$$\begin{aligned}
0={}&\delta_-u_{j,t}
+\partial_x\delta_-\!\left[\frac{u_j^2}{2}+2au_j+h^2\left(\frac{W_j^2}{32}-\frac{W_j}{4}\right)\right]\\
&+\partial_x^2\left(M_-v_j-\frac{h^2}{4}\Delta_h\delta_-u_j\right),\\
0={}&v_{j,t}+\partial_x\!\left[\delta_0\!\left(\frac{u_j^2}{2}+2au_j+h^2\left(\frac{W_j^2}{32}-\frac{W_j}{4}\right)\right)
+(u_j+2a)W_j-4u_j\right]\\
&+\partial_x^2\left[\delta_0u_j+\frac{h^2}{4}\Delta_hW_j\right],
\end{aligned}\tag{8}$$

上述消元在有限 $h$ 下成立；在光滑连续极限中，式（8）趋于

$$u_{yt}+v_{xx}+[(u+2a)u_y]_x=0,\qquad
v_t+u_{xxy}+[(u+2a)v-4u]_x=0.\tag{9}$$

假设 $h\ne0$，$F,G$ 严格为正且关于 $(x,t)$ 联合实解析。`StartPoint` 与证明库中的 `SemiPair` 具有相同定义，由此得到式（7）、（8）。

1. **输入。** `hsp : StartPoint a h F G` 是式（1）的假设，展开后与库中的 `SemiPair` 相同。`hh` 要求 h≠0；`hF/hG` 要求各格点的函数关于 (x,t) 联合实解析，`hFp/hGp` 要求严格为正。

2. **结论。** 第一个 `example` 的冒号后是 `reportN1 = 0 ∧ reportN2 = 0`，对应式（7）。第二个的 `NonlinearPair` 展开为 `n1 = 0 ∧ n2 = 0`，对应式（8）。

3. **使用起点。** 核心定理 `c15_proved` 中的 `rw [(hsp j).1]` 与 `rw [(hsp j).2]` 使用两条双线性方程，使归一化残差 `normA`、`normC` 为零；`c14_proved` 的残差恒等式随后给出 `NonlinearPair`。

4. **得到报告终点。** `semiPair_implies_report8` 用 `physical_v` 对齐 v 的重构后调用核心定理。`semiPair_implies_report7` 使用同一结论：`reportN1_eq_n1` 给出第一条；`c19_proved` 的 W 守恒律经 `omega_conservation_bridge` 变为 `(4/h)·reportN2 = 0`，再由 h≠0 得到第二条。

阅读每个 `example` 时，先看括号中的假设，再看冒号后的结论，最后看 `:=` 后提供证明的定理。这里的 `example` 证明任意满足假设的场；终点运行会检查这两个调用，`#print axioms` 输出它们依赖的基础公理。导数法则、差分恒等式与代数化简封装在导入的证明库中。


In [ ]:
%%lean uw endpoint
import StepUWStart

noncomputable section
open DLWContract DLWContract.ReportUW

-- 式（7）：u 与 ω 的两条演化方程。
example (a h : ℝ) (F G : Lattice) (hh : h ≠ 0)
    (hF : SmoothL F) (hG : SmoothL G) (hFp : PositiveL F) (hGp : PositiveL G)
    (hsp : StartPoint a h F G) :
    reportN1 a h (physU F G) (omega G) = 0 ∧
      reportN2 a h (physU F G) (omega G) = 0 :=
  semiPair_implies_report7 a h F G hh hF hG hFp hGp hsp

-- 式（8）：重构后的物理场 (u, v)。
example (a h : ℝ) (F G : Lattice) (hh : h ≠ 0)
    (hF : SmoothL F) (hG : SmoothL G) (hFp : PositiveL F) (hGp : PositiveL G)
    (hsp : StartPoint a h F G) :
    NonlinearPair a h (physU F G) (vFromOmega h (physU F G) (omega G)) :=
  semiPair_implies_report8 a h F G hh hF hG hFp hGp hsp

#print axioms DLWContract.ReportUW.semiPair_implies_report7
#print axioms DLWContract.ReportUW.semiPair_implies_report8


关键证明封装在 `PkgNonlinear.lean` 与 `ReportNonlinearUW.lean`：`c15_proved` 使用两条双线性假设，将 `normA`、`normC` 置零；`c14_proved` 给出物理场残差恒等式，得到 `NonlinearPair`。`semiPair_implies_report8` 对齐 $v$ 的重构；`semiPair_implies_report7` 再通过 `reportN1_eq_n1` 与 `omega_conservation_bridge` 得到式（7）的两条方程。终点单元调用这些已证明的定理。

### 1.2　保留势的比值形式

本路线仍从式（1）出发，保留势与中心比值，经过下述公式推导到式（21）、（22）。

载入 (Q, R, M) 非线性化的定义、证明及已编译的证明库。


In [ ]:
%%lean qrm import
import ReportNonlinearQRM


在 Lean 中，两条双线性假设合并为 `StartPoint`：


In [ ]:
%%lean qrm start
import StepQRMImport

namespace DLWContract.ReportQRM

def StartPoint (a h : ℝ) (F G : Lattice) : Prop :=
  ∀ j, bil (a-h/2) (F j) (G j) = 0 ∧
    bil (a+h/2) (F j) (G (j+1)) = 0

#print StartPoint

end DLWContract.ReportQRM


保留辅助势 $M_j=\beta_{j,x}$，由式（4）、（5）得

$$\omega_j=M_{j+1}-M_j,\qquad
Z_j=u_j+2(M_j+M_{j+1}).\tag{10}$$

代入式（6），得

$$\begin{aligned}
0={}&u_{j,t}+\partial_x\!\left[
u_{j,x}+\frac{u_j^2}{2}+2au_j
+2(M_j+M_{j+1})_x+\frac{\omega_j^2}{2}-h\omega_j\right],\\
0={}&\omega_{j,t}-\omega_{j,xx}
+\partial_x[(u_j+2a)\omega_j-hu_j],\\
\omega_j={}&M_{j+1}-M_j.
\end{aligned}\tag{11}$$

固定对数分支，定义中心比值

$$Q_j=\exp\!\left(\alpha_j-\frac{\beta_j+\beta_{j+1}}2\right)
=\frac{F_j}{\sqrt{G_jG_{j+1}}},\qquad
u_j=2\frac{Q_{j,x}}{Q_j}.\tag{12}$$

其中 $Q_j$ 位于 $y=(j+\tfrac12)h$，$M_j$ 位于 $y=jh$。由此有

$$u_{j,t}=2\partial_x\!\left(\frac{Q_{j,t}}{Q_j}\right),\qquad
u_{j,x}+\frac{u_j^2}{2}=2\frac{Q_{j,xx}}{Q_j}.\tag{13}$$

式（11）的第一条化为

$$\partial_x\!\left[
\frac{Q_{j,t}+Q_{j,xx}+2aQ_{j,x}}{Q_j}
+(M_j+M_{j+1})_x+\frac{\omega_j^2}{4}-\frac h2\omega_j
\right]=0.\tag{14}$$

由式（12），括号内为 $(A_j+C_j)/2=0$，故

$$\begin{aligned}
0={}&Q_{j,t}+Q_{j,xx}+2aQ_{j,x}\\
&+\left[(M_j+M_{j+1})_x
+\frac{\omega_j^2}{4}-\frac h2\omega_j\right]Q_j,\\
0={}&\omega_{j,t}-\omega_{j,xx}
+\partial_x\!\left[2a\omega_j
+2\frac{Q_{j,x}}{Q_j}(\omega_j-h)\right],\\
\omega_j={}&M_{j+1}-M_j.
\end{aligned}\tag{15}$$

### 1.3　势差的乘积分解

令

$$R_j=\frac{1-\omega_j/h}{Q_j},\qquad
\omega_j=h(1-Q_jR_j),\qquad
\frac{M_{j+1}-M_j}{h}+Q_jR_j=1.\tag{16}$$

在 $Q_j\ne0$ 时，$(Q_{j,x}/Q_j)(\omega_j-h)=-hQ_{j,x}R_j$，且

$$\frac{\omega_j^2}{4}-\frac h2\omega_j
=\frac{h^2}{4}(Q_j^2R_j^2-1).\tag{17}$$

代入式（15），得

$$\begin{aligned}
0={}&Q_{j,t}+Q_{j,xx}+2aQ_{j,x}\\
&+\left[(M_j+M_{j+1})_x
+\frac{h^2}{4}(Q_j^2R_j^2-1)\right]Q_j,\\
0={}&(Q_jR_j)_t-(Q_jR_j)_{xx}
+2a(Q_jR_j)_x+2(Q_{j,x}R_j)_x.
\end{aligned}\tag{18}$$

利用恒等式

$$-(Q_jR_j)_{xx}+2(Q_{j,x}R_j)_x
=Q_{j,xx}R_j-Q_jR_{j,xx}.
\tag{19}$$

第二条可写为

$$R_j(Q_{j,t}+Q_{j,xx}+2aQ_{j,x})
+Q_j(R_{j,t}-R_{j,xx}+2aR_{j,x})=0.\tag{20}$$

再由式（18）的第一条，得到

$$\boxed{\begin{aligned}
0={}&Q_{j,t}+Q_{j,xx}+2aQ_{j,x}\\
&+\left[(M_j+M_{j+1})_x
+\frac{h^2}{4}(Q_j^2R_j^2-1)\right]Q_j,\\[2pt]
0={}&R_{j,t}-R_{j,xx}+2aR_{j,x}\\
&-\left[(M_j+M_{j+1})_x
+\frac{h^2}{4}(Q_j^2R_j^2-1)\right]R_j,\\[2pt]
1={}&\frac{M_{j+1}-M_j}{h}+Q_jR_j.
\end{aligned}}\tag{21}$$

物理场由下式恢复：

$$\boxed{\begin{aligned}
u_j&=2\frac{Q_{j,x}}{Q_j},\qquad
\omega_j=h(1-Q_jR_j)=M_{j+1}-M_j,\\
v_j&=4(1-Q_jR_j)+\frac{u_{j+1}-u_{j-1}}{2h}.
\end{aligned}}\tag{22}$$

沿用相同的正则性、正性与 $h\ne0$ 假设，式（21）由双线性起点推出；式（22）给出物理场的重构恒等式。

1. **式（21）。** `semiPair_implies_report21` 接收 `StartPoint` 及相同的正则性、正性、h≠0 假设，合并 `report_Q_equation`、`report_R_equation` 与 `lattice_constraint`。

2. **证明关系。** Q 方程由归一化双线性残差归零得到；R 方程由乘积残差分解、Q 方程和式（7）的第二条得到。

3. **式（22）。** `report22` 合并 u、ω、v 的重构恒等式；这部分只依赖场的定义、正则性、正性与 h≠0。

冒号后给出结论，`:=` 后调用证明定理；关键证明与依赖保留在导入库中。


In [ ]:
%%lean qrm endpoint
import StepQRMStart

noncomputable section
open DLWContract DLWContract.ReportQRM

-- 式（21）：Q、R 的演化方程与格点约束。
example (a h : ℝ) (F G : Lattice) (hne : h ≠ 0)
    (hF : SmoothL F) (hG : SmoothL G) (hFp : PositiveL F) (hGp : PositiveL G)
    (hsp : StartPoint a h F G) :
    residualQ a h F G = 0 ∧ residualR a h F G = 0 ∧
      ∀ j x t, (M G (j+1) x t-M G j x t)/h + Q F G j x t*R h F G j x t = 1 :=
  semiPair_implies_report21 a h F G hne hF hG hFp hGp hsp

-- 式（22）：由 Q、R、M 重构 u、ω、v。
example (h : ℝ) (F G : Lattice) (hne : h ≠ 0)
    (hF : SmoothL F) (hG : SmoothL G) (hFp : PositiveL F) (hGp : PositiveL G) :
    ∀ j x t,
      physU F G j x t = 2 * lx (Q F G) j x t / Q F G j x t ∧
      omega G j x t = h*(1-Q F G j x t*R h F G j x t) ∧
      omega G j x t = M G (j+1) x t-M G j x t ∧
      physV h F G j x t = 4*(1-Q F G j x t*R h F G j x t) +
        (physU F G (j+1) x t-physU F G (j-1) x t)/(2*h) :=
  report22 h F G hne hF hG hFp hGp

#print axioms DLWContract.ReportQRM.semiPair_implies_report21
#print axioms DLWContract.ReportQRM.report22


关键证明封装在 `ReportNonlinearQRM.lean`：`report_Q_equation` 由双线性起点得到 $Q$ 方程；`report_R_equation` 使用乘积残差分解及式（7）的第二条得到 $R$ 方程；`semiPair_implies_report21` 合并两条演化方程与格点约束。`report22` 合并 $u$、$\omega$、$v$ 的重构恒等式。

## 2　DLW 空间残差与一致性

考察 DLW 的纵向空间截断误差：把连续单孤子代入 SD、SDR 与 FD 的方程，比较有限格距残差与其二阶主系数。这里的残差衡量连续解对离散方程的相容程度，演化误差还取决于时间离散与稳定性。横向与时间导数保持解析，SDR 在相容重构下与 SD 具有相同的理想半离散残差。

取 $K=p+q>0$、$\ell=(p-a)^{-1}+(q+a)^{-1}$、$\Gamma=-(p-a)/(q+a)>0$，令 $z=Kx+\ell y+(q^2-p^2)t+\varphi$、$s(z)=(1+e^{-z})^{-1}$、$f_1(z)=s(z+\log\Gamma)$。同一连续解为

$$u=2K(f_1-s),\qquad v=2K\ell(f_1'+s'),\qquad w=v-u_y=4K\ell s'.$$

以方程左端定义 $\mathcal R_h^S$，则 $\mathcal R_h^S=h^2\tau_y^S+O(h^4)$。第一分量在交错中点展开；第二分量在原节点展开。记 $B(w)=w^2/32-w/4$，由式（8）及其直接差分对照得到

$$\tau_y^{FD}=\begin{pmatrix}v_{xxyy}/12\\u_{xxyyy}/6\end{pmatrix},\qquad
\tau_y^{SD}=\tau_y^{SDR}=\begin{pmatrix}v_{xxyy}/12-u_{xxyyy}/4+B(w)_{xy}\\v_{xxyy}/4-u_{xxyyy}/12+B(w)_{xy}+\tfrac12(u_yu_{yy})_x\end{pmatrix}.$$

所有导数在代码中由 sigmoid 的解析多项式递推求出。令 $\zeta=K\ell$，下列全波形上界来自 $\sup|s'|=1/4$、$\sup|s'''|=1/8$、$\sup|s^{(5)}|=1/4$ 与 $\sup|s''|^2=1/108$：

$$\begin{aligned}
\|\tau_{y,1}^{FD}\|_\infty&\le |\zeta|^3/12,&
\|\tau_{y,2}^{FD}\|_\infty&\le |\zeta|^3/6,\\
\|\tau_{y,1}^{SD}\|_\infty&\le 251|\zeta|^3/864+\zeta^2/8,&
\|\tau_{y,2}^{SD}\|_\infty&\le59|\zeta|^3/96+\zeta^2/8.
\end{aligned}$$

采样最大值取自指定的相位区间，解析上界覆盖整个实相位轴。减半格距时，$\|\mathcal R_h/h^2-\tau_y\|$ 应约缩小至四分之一。推导见 [DLW 误差理论](../report/dlw_error_theory.html)。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from types import SimpleNamespace
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print  # 普通 Python 脚本使用文本表；Colab 使用原生表格输出。

# NumPy 使用 IEEE 754 双精度；各段结果保存在 dlw 中。
dlw = SimpleNamespace(ready={"prepare"})

def dlw_require(stage, message):
    if stage not in dlw.ready:
        raise RuntimeError(message)


取原文参数 A：$(a,p,q)=(2,1,2)$，相位区间为 $[-10,10]$。纵向格距从 $h=1/8$ 开始逐次减半，横向和时间导数保持解析。

In [ ]:
dlw_require("prepare", "先运行准备环境。")
dlw.ready = {"prepare"}
# 孤子参数 (a,p,q)：A=(2,1,2)，B=(2,4,-3)。
# h：y 方向格距；z：无量纲相位；x、t 导数保持解析。
dlw_cfg = dict(a=2, p=1, q=2, h=1/8,
               zMin=-10, zMax=10, points=1601, levels=3)
a, p, q, h, zMin, zMax = (dlw_cfg[k] for k in
                          ("a", "p", "q", "h", "zMin", "zMax"))
points, levels = dlw_cfg["points"], dlw_cfg["levels"]
if not np.isfinite([a, p, q, h, zMin, zMax]).all():
    raise ValueError("参数必须是有限实数。")
if not (h > 0 and zMax > zMin and zMax-zMin <= 100):
    raise ValueError("要求 h>0、zMax>zMin，且相位区间宽度≤100。")
if (not isinstance(points, int) or not 41 <= points <= 10001 or
        not isinstance(levels, int) or not 1 <= levels <= 6):
    raise ValueError("points 应在 41—10001；levels 应在 1—6，均取整数。")
if h / 2**(levels-1) < 1/1024:
    raise ValueError("最细格距须≥1/1024，以限制浮点消减误差。")
K, P, Q = p+q, p-a, q+a
if not (K > 0 and P != 0 and Q != 0 and -P/Q > 0):
    raise ValueError("单孤子正分支要求 p+q>0，(p-a)(q+a)<0。")
ell, Gamma, Omega = 1/P + 1/Q, -P/Q, q*q-p*p
zeta = K*ell
if not np.isfinite([ell, Gamma, Omega, zeta]).all():
    raise ValueError("参数过近谱极点，所得系数非有限。")
dlw.cfg = dlw_cfg.copy()
dlw.a, dlw.K, dlw.ell, dlw.Gamma = a, K, ell, Gamma
dlw.Omega, dlw.zeta, dlw.h, dlw.levels = Omega, zeta, h, levels
dlw.zs = zMin + (zMax-zMin)*np.arange(points, dtype=np.float64)/(points-1)
dlw.ready = {"prepare", "config"}
display(pd.DataFrame([[K, ell, Gamma, zeta, h, points]],
                     columns=["K", "ℓ", "Γ", "ζ = Kℓ", "h", "相位点数"]))


sigmoid 的各阶导数写成 $s(1-s)$ 与多项式的乘积。这一递推同时给出连续剖面和残差主系数所需的解析导数。

In [ ]:
dlw_require("config", "先运行谱参数配置。")
dlw.ready = {"prepare", "config"}
# s^(n)=s(1-s)*P_n(s)，P_(n+1)=(1-2s)P_n+s(1-s)P_n'。
dlw_polys = [np.array([1.0])]
for n in range(1, 5):
    c = dlw_polys[-1]
    dlw_polys.append(np.array([(k+1)*((c[k] if k < len(c) else 0)
                                    -(c[k-1] if k else 0))
                              for k in range(len(c)+1)]))

def dlw_sigmoid_jet(z):
    z = np.asarray(z, dtype=np.float64)
    e = np.exp(-np.abs(z))
    s = np.where(z >= 0, 1/(1+e), e/(1+e))
    theta = e/(1+e)**2
    return np.array([s] + [theta*np.polynomial.polynomial.polyval(s, c)
                          for c in dlw_polys])

def dlw_profile(z):
    s, f = dlw_sigmoid_jet(z), dlw_sigmoid_jet(z+np.log(dlw.Gamma))
    # u[n]、v[n] 表示对 z 的 n 阶导数，支持标量或整列相位。
    return dict(s=s, f=f, u=2*dlw.K*(f[:4]-s[:4]),
                v=2*dlw.K*dlw.ell*(f[1:4]+s[1:4]))

dlw.profile = dlw_profile
dlw.ready = {"prepare", "config", "profile"}


将连续剖面代入截断展开，得到 SD 与 FD 的二阶主系数。下列上界使用 sigmoid 导数在全部实相位上的极值。

In [ ]:
dlw_require("profile", "先运行连续剖面。")
dlw.ready = {"prepare", "config", "profile"}

def dlw_coefficient(z):
    c = dlw.profile(z)
    s, f, zeta = c["s"], c["f"], dlw.zeta
    d1, d2, d3 = f[1]-s[1], f[2]-s[2], f[3]-s[3]
    # B=(s″)²+s′*s‴=0.5*((s′)²)″。
    B = s[2]**2 + s[1]*s[3]
    return {
        "FD": np.array([zeta**3*(f[5]+s[5])/6,
                        zeta**3*(f[5]-s[5])/3]),
        "SD": np.array([zeta**3*((2*s[5]-f[5])/3+B)-zeta**2*s[3],
                        zeta**3*((f[5]+2*s[5])/3+B+2*(d2*d2+d1*d3))
                        -zeta**2*s[3]])
    }

# 解析上界覆盖全部实相位 z，不依赖相位区间与采样点数。
Z = abs(dlw.zeta)
dlw.bounds = {"FD": np.array([Z**3/12, Z**3/6]),
              "SD": np.array([251*Z**3/864+Z**2/8,
                              59*Z**3/96+Z**2/8])}
dlw.coefficient = dlw_coefficient
dlw.ready = {"prepare", "config", "profile", "coefficient"}


有限格距残差直接由离散方程左端求值。第一条方程取交错中点，第二条取原节点；每个偏移节点均按 $W=v-\delta_0u$ 重构。

In [ ]:
dlw_require("coefficient", "先运行二阶主系数。")
dlw.ready = {"prepare", "config", "profile", "coefficient"}

def dlw_residual(z, hh, scheme):
    if scheme not in ("SD", "FD"):
        raise ValueError("scheme 应为 SD 或 FD。")
    z = np.asarray(z, dtype=np.float64)
    a, K, ell, Omega = dlw.a, dlw.K, dlw.ell, dlw.Omega
    cache = {}

    def node(j):
        if j in cache:
            return cache[j]
        c = dlw.profile(z+ell*hh*j)
        plus = dlw.profile(z+ell*hh*(j+1))
        minus = dlw.profile(z+ell*hh*(j-1))
        # 每个偏移节点重新构造 D=δ₀u、W=v−δ₀u。
        D = (plus["u"][:3]-minus["u"][:3])/(2*hh)
        W = c["v"]-D
        u = c["u"]
        Ax = K*(u[0]+2*a)*u[1]
        Hx = Ax + (hh*hh*K*(W[0]/16-1/4)*W[1] if scheme == "SD" else 0)
        out = dict(c, D=D, W=W, Ax=Ax, Hx=Hx)
        cache[j] = out
        return out

    # 第一式在交错中点，第二式在节点 j=0；x、t 导数保持解析。
    plus, minus, c = node(1/2), node(-1/2), node(0)
    r1 = Omega*(plus["u"][1]-minus["u"][1])/hh
    r1 += (plus["Hx"]-minus["Hx"])/hh
    r1 += K*K*((plus["u"][2]-minus["u"][2])/hh
               +(plus["W"][2]+minus["W"][2])/2 if scheme == "SD"
               else (plus["v"][2]+minus["v"][2])/2)
    u, v, W, D = c["u"], c["v"], c["W"], c["D"]
    r2 = Omega*v[1]
    if scheme == "SD":
        pp, mm = node(1), node(-1)
        r2 += ((pp["Hx"]-mm["Hx"])/(2*hh)
               +K*(u[1]*W[0]+(u[0]+2*a)*W[1]-4*u[1])
               +K*K*(D[2]+(pp["W"][2]-2*W[2]+mm["W"][2])/4))
    else:
        r2 += K*(u[1]*v[0]+(u[0]+2*a)*v[1]-4*u[1])+K*K*D[2]
    return np.array([r1, r2])

dlw.residual = dlw_residual
dlw.ready = {"prepare", "config", "profile", "coefficient", "residual"}


比较采样主系数与全波形解析上界，并检查同一剖面对连续 DLW 方程的残差。

In [ ]:
dlw_require("residual", "先运行有限格距残差。")
dlw.ready = {"prepare", "config", "profile", "coefficient", "residual"}
dlw.tau = dlw.coefficient(dlw.zs)
c = dlw.profile(dlw.zs)
u, v, b = c["u"], c["v"], c["u"][0]+2*dlw.a
K, ell, Omega = dlw.K, dlw.ell, dlw.Omega
terms1 = np.array([ell*Omega*u[2], K*K*v[2],
                   K*ell*(u[1]*u[1]+b*u[2])])
terms2 = np.array([Omega*v[1], K*K*ell*u[3],
                   K*(u[1]*v[0]+b*v[1]-4*u[1])])
# 两条连续方程的归一化残差 abs(sum)/(1+sum(abs))。
dlw.continuumDefect = max(float(np.max(np.abs(terms.sum(axis=0))
                                     /(1+np.abs(terms).sum(axis=0))))
                          for terms in (terms1, terms2))
if dlw.continuumDefect > 1e-10:
    raise ArithmeticError("连续 DLW 方程核对失败，请检查剖面或谱参数。")
dlw.coeffRows = []
for S in ("SD", "FD"):
    for i in range(2):
        sampled = float(np.max(np.abs(dlw.tau[S][i])))
        bound = float(dlw.bounds[S][i])
        if not np.isfinite(sampled) or sampled > bound*(1+1e-10):
            raise ArithmeticError(f"{S} 第 {i+1} 式超出解析主系数上界。")
        dlw.coeffRows.append([f"{S}{' / SDR' if S == 'SD' else ''} · {i+1}",
                              sampled, bound, dlw.h**2*sampled, dlw.h**2*bound])
display(pd.DataFrame(dlw.coeffRows, columns=["方程", "采样 max |τ|", "全波形解析上界",
                                           "采样 h² max |τ|", "h² 解析上界"]))
print("continuumDefect =", dlw.continuumDefect)
dlw.ready = {"prepare", "config", "profile", "coefficient", "residual", "main"}


逐次减半格距，比较 $\mathcal R_h/h^2$ 与 $\tau_y$。表中的收敛阶来自相邻格距的差值，曲线展示最细格距下的缩放残差与主系数。

In [ ]:
dlw_require("main", "先运行主系数计算。")
dlw.ready = {"prepare", "config", "profile", "coefficient", "residual", "main"}
# R_h/h²−τ=O(h²)：相邻两档误差之比应趋近 4。
dlw.convergence, dlw.finest, previous = [], {}, {}
for level in range(dlw.levels):
    hh = dlw.h/2**level
    for S in ("SD", "FD"):
        values = dlw.residual(dlw.zs, hh, S)/(hh*hh)
        if not np.isfinite(values).all():
            raise ArithmeticError("有限 h 残差非有限，请检查参数。")
        errors = values-dlw.tau[S]
        for i in range(2):
            err = float(np.max(np.abs(errors[i])))
            old = previous.get((S, i))
            rate = np.log2(old/err) if old is not None and old > 0 and err > 0 else np.nan
            dlw.convergence.append([f"{S} · {i+1}", hh,
                                    float(np.max(np.abs(values[i]))), err, rate])
            previous[S, i] = err
        if level == dlw.levels-1:
            dlw.finest[S] = values
display(pd.DataFrame(dlw.convergence, columns=["方程", "h", "采样 max |R_h / h²|",
                                             "采样 max |R_h / h² − τ|", "减半收敛阶"]))
for i in range(2):
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(dlw.zs, dlw.tau["SD"][i], label=r"SD / SDR $\tau$")
    ax.plot(dlw.zs, dlw.tau["FD"][i], label=r"FD $\tau$")
    hh = dlw.h/2**(dlw.levels-1)
    ax.plot(dlw.zs, dlw.finest["SD"][i], "--", label=f"SD h={hh:g}")
    ax.plot(dlw.zs, dlw.finest["FD"][i], "--", label=f"FD h={hh:g}")
    ax.set(title=f"DLW equation {i+1}", xlabel="Phase z", ylabel=r"Residual / $h^2$")
    ax.grid(alpha=.2)
    ax.legend()
    fig.tight_layout()
    plt.show()
dlw.ready = {"prepare", "config", "profile", "coefficient", "residual", "main", "convergence"}


式（7）、（8）与式（21）、（22）给出同一双线性起点的两种非线性表示。空间残差比较反映这些方程在连续单孤子上的二阶相容性；不同参数下的系数大小与演化至有限时间后的场误差需要分别考察。

孤子演化的数值比较见 [DLW 数值分析](DLW数值分析report.ipynb)。